In [ ]:
from pathlib import Path
import json,hashlib,zipfile
ROOT=Path.cwd()
while not (ROOT/'pyproject.toml').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Run inside repository')
    ROOT=ROOT.parent
D=ROOT/'docs/ntu-completion-20261011/stage5'
delta=json.loads((D/'候選增量.json').read_text())
snapshot=json.loads((D/'結構化候選資料.json').read_text())
assert snapshot['candidate_only'] and not snapshot['research_meta']['runtime_materialization_performed']
assert len(delta['observations'])==51 and len(snapshot['schools'])==288
assert sum(len(s['observations']) for s in snapshot['schools'])==659
print({'combined_candidates':659,'incremental_candidates':51,'runtime_imported':False})


In [ ]:
manifest=json.loads((D/'已讀原文manifest.json').read_text())
normalize=lambda text:' '.join(text.split())
with zipfile.ZipFile(D/manifest['archive_file']) as z:
    assert z.testzip() is None
    assert hashlib.sha256((D/manifest['archive_file']).read_bytes()).hexdigest()==manifest['archive_sha256']
    sources={s['source_id']:s for s in manifest['source_bodies']}
    for source in sources.values():
        assert hashlib.sha256(z.read(source['archive_body_path'])).hexdigest()==source['sha256']
    for observation in delta['observations']:
        sid=observation['source_id']
        assert normalize(observation['original_quote']) in normalize(z.read('texts/'+sid+'.txt').decode())
        assert observation['raw_body_sha256']==sources[sid]['sha256']
        if observation['pdf_page']:
            pages=json.loads(z.read('texts/'+sid+'.pages.json'))
            assert normalize(observation['original_quote']) in normalize(pages[observation['pdf_page']-1]['text'])
assert len(sources)==40
print({'raw_bodies_verified':40,'quoted_observations_verified':51})


In [ ]:
from openpyxl import load_workbook
progress=json.loads((D/'priority-review-progress.json').read_text())
assert progress['fixed_priority_field_denominator']==501
assert progress['completed_original_cached_body_sets']==120
assert progress['remaining_original_cached_body_sets']==381
book=load_workbook(D/'288校逐欄查核_可回填.xlsx',read_only=True,data_only=False)
assert len(book.sheetnames)==11 and book['288校逐欄回填'].max_row==4033
assert sum(r[5] is True for r in book['固定501項正文進度'].iter_rows(min_row=2,values_only=True))==120
assert not any(c.data_type=='f' for s in book for r in s for c in r)
book.close()
print({'school_field_rows':4032,'fixed_priority_completed':120,'fixed_priority_total':501,'current_cycle_fully_verified_fields':0})
